In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from torchinfo import summary

In [29]:
data=pd.DataFrame(pd.read_csv('data.csv'))
data.head(10)

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1
5,6,2990,77.417073,50.954344,0.752861,3080,61.700780,0.584898,216.930,0.798439,1.519342,1
6,7,3556,84.323564,55.413061,0.753762,3636,67.287739,0.750211,227.007,0.867148,1.521727,1
7,8,3788,86.952411,56.444769,0.760664,3866,69.448048,0.800676,235.476,0.858473,1.540487,1
8,9,2629,74.133114,48.074144,0.761228,2790,57.856260,0.640595,207.325,0.768594,1.542058,1
9,10,5719,106.721142,68.977700,0.763053,5819,85.332625,0.754983,281.839,0.904748,1.547183,1


In [30]:
# for column in data.columns:
#     data[column] = data[column]/data[column].abs().max() # Divide by the maximum of the column which will make max value of each column is 1
# data.head()
     

In [31]:
features=[i for i in data]
train_features=features[1:len(features)-1]
target_features=[features[-1]]

In [32]:
X=np.array(data[train_features].copy())
y=np.array(data[target_features].copy())

In [33]:
X_train, X_test,y_train, y_test = train_test_split(X,y ,
                                   random_state=104, 
                                   test_size=0.25, 
                                   shuffle=True)

In [34]:
class data_set:
	def __init__(self, X, y):
		self.X=torch.tensor(X, dtype=torch.float32)
		self.y=torch.tensor(y, dtype=torch.float32)
	
	def __len__(self):
		return len(self.y)
	
	def __getitem__(self, index):
		return self.X[index], self.y[index]

In [35]:
train_data=data_set(X_train, y_train)
test_data=data_set(X_test, y_test)

In [36]:
train_data_loader=DataLoader(train_data, batch_size=8, shuffle=True)
test_data_loader=DataLoader(test_data, batch_size=8)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.input_layer=nn.Linear(X.shape[1],64)
        self.relu1=nn.ReLU()
        self.linear1=nn.Linear(64,128)
        self.relu2=nn.ReLU()
        self.linear2=nn.Linear(128,64)
        self.relu3=nn.ReLU()
        self.linear3=nn.Linear(64,1)
        self.out=nn.Sigmoid()
    
    def forward(self, x):
        x=self.input_layer(x)
        x=self.linear1(x)
        x=self.relu(x)
        x=self.linear2(x)
        x=self.relu(x)
        x=self.linear3(x)
        x=self.relu(x)
        x=self.out(x)
        return x

In [38]:
model=Net()
summary(model)

Layer (type:depth-idx)                   Param #
Net                                      --
├─ReLU: 1-1                              --
├─Linear: 1-2                            704
├─Linear: 1-3                            8,320
├─Linear: 1-4                            8,256
├─Linear: 1-5                            65
├─Sigmoid: 1-6                           --
Total params: 17,345
Trainable params: 17,345
Non-trainable params: 0

In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [40]:
EPOCH=10
for e in range(EPOCH):
	model.train()
	for i, (inputs, labels) in enumerate(train_data_loader):
		optimizer.zero_grad()
		preds=model(inputs)
		loss=criterion(preds, labels)
		loss.backward()
		optimizer.step()

In [41]:
gud=0
for input, label in train_data:
	pred=model(input)
	print(pred)
	real_label=0 if pred<0.5 else 1
	if(pred==label):
		gud+=1

print(gud/train_data.__len__())

tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], grad_fn=<SigmoidBackward0>)
tensor([1.], gra